In [3]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pickle
from pathlib import Path
import os

# STEP 1: Load Trained Model and Data
print("\n" + "="*60)
print("STEP 1: Loading Model and Data")
print("="*60)

model_path = Path('../models/activity_model.h5')
model = keras.models.load_model(model_path)
print(f"Loaded model from: {model_path}")

with open('../data/processed/preprocessing_info.pkl', 'rb') as f:
    preprocessing_info = pickle.load(f)

X_test = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')
X_train = np.load('../data/processed/X_train.npy')

print(f"Train data: {X_train.shape}")
print(f"Test data: {X_test.shape}")

# STEP 2: Evaluate Original Model
print("\n" + "="*60)
print("STEP 2: Original Model Performance")
print("="*60)

loss_original, acc_original = model.evaluate(X_test, y_test, verbose=0)
print(f"Original Model:")
print(f"  Accuracy: {acc_original*100:.2f}%")
print(f"  Loss: {loss_original:.4f}")

model_size_original = os.path.getsize(model_path) / 1024
print(f"  File size: {model_size_original:.2f} KB")

# STEP 3: Representative dataset for calibration
print("\n" + "="*60)
print("STEP 3: Preparing Calibration Data")
print("="*60)

def representative_dataset_gen():
    for i in range(len(X_train)):
        sample = X_train[i:i+1].astype(np.float32)
        yield [sample]

print(f"Using ALL {len(X_train)} training samples for calibration")

# STEP 4: Convert — Try BOTH full INT8 and hybrid (INT8 weights, float I/O)
print("\n" + "="*60)
print("STEP 4: Converting to TFLite — Two Variants")
print("="*60)

results = {}

# --- Variant A: Full INT8 (input + output INT8) ---
print("\n--- Variant A: Full INT8 (input/output INT8) ---")
converter_a = tf.lite.TFLiteConverter.from_keras_model(model)
converter_a.optimizations = [tf.lite.Optimize.DEFAULT]
converter_a.representative_dataset = representative_dataset_gen
converter_a.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_a.inference_input_type = tf.int8
converter_a.inference_output_type = tf.int8

tflite_full_int8 = converter_a.convert()
path_full = Path('../models/activity_model_full_int8.tflite')
with open(path_full, 'wb') as f:
    f.write(tflite_full_int8)
print(f"  Saved: {path_full} ({len(tflite_full_int8)/1024:.1f} KB)")

# --- Variant B: Hybrid (INT8 weights, float input/output) ---
print("\n--- Variant B: Hybrid (INT8 weights, float I/O) ---")
converter_b = tf.lite.TFLiteConverter.from_keras_model(model)
converter_b.optimizations = [tf.lite.Optimize.DEFAULT]
converter_b.representative_dataset = representative_dataset_gen
converter_b.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_b.inference_input_type = tf.float32
converter_b.inference_output_type = tf.float32

tflite_hybrid = converter_b.convert()
path_hybrid = Path('../models/activity_model_hybrid.tflite')
with open(path_hybrid, 'wb') as f:
    f.write(tflite_hybrid)
print(f"  Saved: {path_hybrid} ({len(tflite_hybrid)/1024:.1f} KB)")

# STEP 5: Test Both Variants
print("\n" + "="*60)
print("STEP 5: Testing Both Quantized Models")
print("="*60)

def test_tflite_model(model_path, X_test, y_test):
    """Test a TFLite model and return accuracy."""
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    input_dtype = input_details[0]['dtype']
    output_dtype = output_details[0]['dtype']
    
    print(f"  Input: {input_details[0]['shape']}, dtype={input_dtype.__name__}")
    print(f"  Output: {output_details[0]['shape']}, dtype={output_dtype.__name__}")
    
    if input_dtype == np.int8:
        input_scale = input_details[0]['quantization'][0]
        input_zp = input_details[0]['quantization'][1]
        print(f"  Input quant: scale={input_scale:.6f}, zp={input_zp}")
    
    if output_dtype == np.int8:
        output_scale = output_details[0]['quantization'][0]
        output_zp = output_details[0]['quantization'][1]
        print(f"  Output quant: scale={output_scale:.6f}, zp={output_zp}")
    
    predictions = []
    for i in range(len(X_test)):
        input_data = X_test[i:i+1].astype(np.float32)
        
        if input_dtype == np.int8:
            input_data = (input_data / input_scale + input_zp).astype(np.int8)
        
        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()
        
        output_data = interpreter.get_tensor(output_details[0]['index'])
        
        if output_dtype == np.int8:
            output_data = (output_data.astype(np.float32) - output_zp) * output_scale
        
        predictions.append(np.argmax(output_data))
    
    predictions = np.array(predictions)
    accuracy = np.mean(predictions == y_test)
    return accuracy, predictions

# Test Variant A: Full INT8
print("\n--- Variant A: Full INT8 ---")
acc_full, preds_full = test_tflite_model(path_full, X_test, y_test)
drop_full = (acc_original - acc_full) * 100
print(f"  Accuracy: {acc_full*100:.2f}% (drop: {drop_full:.2f}%)")

# Test Variant B: Hybrid
print("\n--- Variant B: Hybrid (float I/O) ---")
acc_hybrid, preds_hybrid = test_tflite_model(path_hybrid, X_test, y_test)
drop_hybrid = (acc_original - acc_hybrid) * 100
print(f"  Accuracy: {acc_hybrid*100:.2f}% (drop: {drop_hybrid:.2f}%)")

# Pick the best variant
if acc_hybrid >= acc_full:
    best_name = "Hybrid (float I/O)"
    best_acc = acc_hybrid
    best_drop = drop_hybrid
    best_path = path_hybrid
    best_model = tflite_hybrid
    best_preds = preds_hybrid
else:
    best_name = "Full INT8"
    best_acc = acc_full
    best_drop = drop_full
    best_path = path_full
    best_model = tflite_full_int8
    best_preds = preds_full

# STEP 6: Comparison Summary
print("\n" + "="*60)
print("STEP 6: Comparison Summary")
print("="*60)

print(f"{'Metric':<25} {'Float32':<18} {'Full INT8':<18} {'Hybrid':<18}")
print("-"*79)
print(f"{'Format':<25} {'Keras (.h5)':<18} {'INT8 in/out':<18} {'INT8 w/ float IO':<18}")
print(f"{'File Size':<25} {f'{model_size_original:.1f} KB':<18} {f'{len(tflite_full_int8)/1024:.1f} KB':<18} {f'{len(tflite_hybrid)/1024:.1f} KB':<18}")
print(f"{'Test Accuracy':<25} {f'{acc_original*100:.2f}%':<18} {f'{acc_full*100:.2f}%':<18} {f'{acc_hybrid*100:.2f}%':<18}")
print(f"{'Accuracy Drop':<25} {'-':<18} {f'{drop_full:.2f}%':<18} {f'{drop_hybrid:.2f}%':<18}")

print(f"\n>>> Best variant: {best_name} ({best_acc*100:.2f}%, drop: {best_drop:.2f}%)")

# STEP 7: Generate deployment files from best variant
print("\n" + "="*60)
print("STEP 7: Generating Deployment Files (from best variant)")
print("="*60)

# Copy best model as the canonical quantized model
tflite_path = Path('../models/activity_model_quantized.tflite')
with open(tflite_path, 'wb') as f:
    f.write(best_model)
print(f"Saved: {tflite_path} ({len(best_model)/1024:.1f} KB)")

# C header
def create_c_array(data, var_name):
    c_array = f"// Auto-generated file\n"
    c_array += f"// TensorFlow Lite model for activity recognition\n"
    c_array += f"// Model size: {len(data)} bytes\n\n"
    c_array += f"alignas(8) const unsigned char {var_name}[] = {{\n"
    for i in range(0, len(data), 12):
        row = data[i:i+12]
        c_array += "  " + ", ".join([f"0x{b:02x}" for b in row])
        if i + 12 < len(data):
            c_array += ","
        c_array += "\n"
    c_array += f"}};\n"
    c_array += f"const unsigned int {var_name}_len = {len(data)};\n"
    return c_array

c_header = create_c_array(best_model, "activity_model_data")
c_header_path = Path('../models/model_data.h')
with open(c_header_path, 'w') as f:
    f.write(c_header)
print(f"Saved: {c_header_path} ({len(best_model)} bytes)")

# STEP 8: Create Config Header for C++
print("\n" + "="*60)
print("STEP 8: Creating C++ Config Header")
print("="*60)

num_classes = len(preprocessing_info['label_to_activity'])
num_features = len(preprocessing_info['mean'])

if 'config' in preprocessing_info and isinstance(preprocessing_info['config'], dict):
    window_size = preprocessing_info['config'].get('window_size', 128)
    sampling_rate = preprocessing_info['config'].get('sampling_rate', 50)
else:
    window_size = 128
    sampling_rate = 50

print(f"  NUM_CLASSES: {num_classes}")
print(f"  NUM_FEATURES: {num_features}")
print(f"  WINDOW_SIZE: {window_size}")
print(f"  SAMPLING_RATE: {sampling_rate}")

config_content = f"""// Auto-generated configuration file
// Generated from Python preprocessing

#ifndef CONFIG_H
#define CONFIG_H

// Model configuration
#define NUM_CLASSES {num_classes}
#define NUM_FEATURES {num_features}
#define WINDOW_SIZE {window_size}
#define SAMPLING_RATE {sampling_rate}

// Normalization parameters (from training data)
const float SENSOR_MEAN[NUM_FEATURES] = {{
    {', '.join([f'{m:.6f}f' for m in preprocessing_info['mean']])}
}};

const float SENSOR_STD[NUM_FEATURES] = {{
    {', '.join([f'{s:.6f}f' for s in preprocessing_info['std']])}
}};

// Activity labels
const char* ACTIVITY_LABELS[NUM_CLASSES] = {{
    {', '.join([f'"{preprocessing_info["label_to_activity"][i]}"' for i in range(num_classes)])}
}};

// Sensor column order (for reference)
// 0: Ax, 1: Ay, 2: Az, 3: Gx, 4: Gy, 5: Gz, 6: Acc_mag, 7: Gyro_mag

#endif // CONFIG_H
"""

config_path = Path('../models/config.h')
with open(config_path, 'w') as f:
    f.write(config_content)

print(f"Saved: {config_path}")

print("\n" + "="*60)
print("QUANTIZATION COMPLETE!")
print("="*60)
print(f"  Original: {acc_original*100:.2f}% accuracy, {model_size_original:.1f} KB")
print(f"  Best quantized ({best_name}): {best_acc*100:.2f}% accuracy, {len(best_model)/1024:.1f} KB")
print(f"  Accuracy drop: {best_drop:.2f}%")
print(f"\nFiles: model_data.h, config.h, activity_model_quantized.tflite")
print(f"Status: {'READY FOR DEPLOYMENT' if best_drop < 5 else 'Check accuracy drop'}")


STEP 1: Loading Model and Data
Loaded model from: ../models/activity_model.h5
Train data: (7162, 600, 8)
Test data: (638, 600, 8)

STEP 2: Original Model Performance


2026-06-22 11:01:29.051066: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


Original Model:
  Accuracy: 95.14%
  Loss: 0.1881
  File size: 391.40 KB

STEP 3: Preparing Calibration Data
Using ALL 7162 training samples for calibration

STEP 4: Converting to TFLite — Two Variants

--- Variant A: Full INT8 (input/output INT8) ---
INFO:tensorflow:Assets written to: /var/folders/52/bpf46hx95sl2yr7m61mcmnlw0000gn/T/tmppj8r40qx/assets


INFO:tensorflow:Assets written to: /var/folders/52/bpf46hx95sl2yr7m61mcmnlw0000gn/T/tmppj8r40qx/assets
/opt/homebrew/Caskroom/miniforge/base/envs/tinyml2/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:887: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-06-22 11:01:30.373028: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
2026-06-22 11:01:30.373064: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-06-22 11:01:30.373581: I tensorflow/cc/saved_model/reader.cc:45] Reading SavedModel from: /var/folders/52/bpf46hx95sl2yr7m61mcmnlw0000gn/T/tmppj8r40qx
2026-06-22 11:01:30.374670: I tensorflow/cc/saved_model/reader.cc:91] Reading meta graph with tags { serve }
2026-06-22 11:01:30.374675: I tensorflow/cc/saved_model/reader.cc:132] Reading SavedModel debug info (if present) from: /var/folders/52/bpf4

  Saved: ../models/activity_model_full_int8.tflite (40.5 KB)

--- Variant B: Hybrid (INT8 weights, float I/O) ---
INFO:tensorflow:Assets written to: /var/folders/52/bpf46hx95sl2yr7m61mcmnlw0000gn/T/tmp264n3g13/assets


INFO:tensorflow:Assets written to: /var/folders/52/bpf46hx95sl2yr7m61mcmnlw0000gn/T/tmp264n3g13/assets
/opt/homebrew/Caskroom/miniforge/base/envs/tinyml2/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:887: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-06-22 11:01:56.891440: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
2026-06-22 11:01:56.891456: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-06-22 11:01:56.891611: I tensorflow/cc/saved_model/reader.cc:45] Reading SavedModel from: /var/folders/52/bpf46hx95sl2yr7m61mcmnlw0000gn/T/tmp264n3g13
2026-06-22 11:01:56.892704: I tensorflow/cc/saved_model/reader.cc:91] Reading meta graph with tags { serve }
2026-06-22 11:01:56.892709: I tensorflow/cc/saved_model/reader.cc:132] Reading SavedModel debug info (if present) from: /var/folders/52/bpf4

  Saved: ../models/activity_model_hybrid.tflite (40.8 KB)

STEP 5: Testing Both Quantized Models

--- Variant A: Full INT8 ---
  Input: [  1 600   8], dtype=int8
  Output: [1 8], dtype=int8
  Input quant: scale=0.193294, zp=-8
  Output quant: scale=0.003906, zp=-128
  Accuracy: 80.25% (drop: 14.89%)

--- Variant B: Hybrid (float I/O) ---
  Input: [  1 600   8], dtype=float32
  Output: [1 8], dtype=float32
  Accuracy: 87.30% (drop: 7.84%)

STEP 6: Comparison Summary
Metric                    Float32            Full INT8          Hybrid            
-------------------------------------------------------------------------------
Format                    Keras (.h5)        INT8 in/out        INT8 w/ float IO  
File Size                 391.4 KB           40.5 KB            40.8 KB           
Test Accuracy             95.14%             80.25%             87.30%            
Accuracy Drop             -                  14.89%             7.84%             

>>> Best variant: Hybrid (float I/

fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32
